# 1. Coleta de Dados Automatizada (Web Scraping)

**Objetivo:** Coletar dados públicos de Smart TVs no e-commerce Mercado Livre para responder à pergunta de negócio: *"Qual marca possui os preços mais em conta?"*.

**Escopo:**
* Fonte: Mercado Livre (Busca: Smart TV)
* Meta mínima: 30 produtos/ofertas
* Campos coletados: Nome do produto, preço, loja/fonte, URL e categoria

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

print("✅ Bibliotecas importadas com sucesso!")

✅ Bibliotecas importadas com sucesso!


In [ ]:
# 1. Definir a URL da API do Mercado Livre para a busca de Smart TVs

url = "https://lista.mercadolivre.com.br/smart-tv"

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7"
}

In [ ]:
response = requests.get(url, headers=headers)

In [ ]:
import os
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

print("✅ Bibliotecas importadas com sucesso!")

✅ Bibliotecas importadas com sucesso!


In [ ]:
# URL da categoria de TVs no Buscapé
url_buscape = "https://www.buscape.com.br/tv"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"
}

response = requests.get(url_buscape, headers=headers)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, "html.parser")
    # Busca por elementos de card que contêm informação do produto
    cards = soup.select('[data-testid="product-card"], a[class*="ProductCard"]')
    
    # Filtra cards válidos com conteúdo textual expressivo
    cards_validos = [c for c in cards if len(c.get_text(strip=True)) > 10]
    print(f"✅ Conexão estabelecida com sucesso (HTTP {response.status_code})!")
    print(f"Total de produtos válidos encontrados na página: {len(cards_validos)}")
else:
    print(f"❌ Erro ao conectar com o Buscapé: HTTP Status {response.status_code}")

✅ Conexão estabelecida com sucesso (HTTP 200)!
Total de produtos válidos encontrados na página: 44


In [ ]:
lista_produtos = []

for index, card in enumerate(cards_validos, start=1):
    texto_completo = card.get_text(separator=" | ", strip=True)
    
    # Extração de Preço via Expressão Regular
    preco_match = re.search(r'R\$\s?[\d\.]+(?:,\d{2})?', texto_completo)
    preco_str = preco_match.group(0) if preco_match else "N/A"
    
    # Extração do Nome do Produto
    partes = [p.strip() for p in texto_completo.split("|") if len(p.strip()) > 3 and "R$" not in p]
    nome = partes[0] if partes else "N/A"
    
    # Extração da Loja Parceira
    loja = "N/A"
    for parte in partes:
        if "Via" in parte or ("em" in parte and "lojas" in parte):
            loja = parte
            break
            
    # Extração do Link do Produto
    link = card.get("href") if card.has_attr("href") else "N/A"
    if link != "N/A" and not link.startswith("http"):
        link = "https://www.buscape.com.br" + link

    lista_produtos.append({
        "id_coleta": index,
        "produto": nome,
        "preco_bruto": preco_str,
        "loja_oferta": loja,
        "url": link,
        "categoria": "Smart TV"
    })

# Converter para DataFrame do Pandas
df_bruto = pd.DataFrame(lista_produtos)
print(f"✅ Processamento concluído: {len(df_bruto)} registros estruturados.")

✅ Processamento concluído: 44 registros estruturados.


In [ ]:
# Garantir existência da pasta 'data' na raiz do projeto
os.makedirs("../data", exist_ok=True)

# Salvando a base de dados bruta
caminho_csv = "../data/dados_brutos.csv"
df_bruto.to_csv(caminho_csv, index=False, encoding="utf-8-sig")

print(f"💾 Arquivo gerado e salvo com sucesso em: {caminho_csv}\n")

# Visualização das primeiras linhas
df_bruto.head()

💾 Arquivo gerado e salvo com sucesso em: ../data/dados_brutos.csv



,id_coleta,produto,preco_bruto,loja_oferta,url,categoria
0,1,"Smart TV Mini LED 55"" TCL 4K 55C6K","R$ 3.279,00",Via Amazon,N/A,Smart TV
1,2,"Smart TV Mini LED 55"" TCL 4K 55C6K","R$ 3.279,00",Via Amazon,https://www.buscape.com.br/tv/smart-tv-mini-le...,Smart TV
2,3,"Smart TV QD-Mini LED 65"" TCL 4K 65C6K","R$ 3.963,60",Via Webcontinental,N/A,Smart TV
3,4,"Smart TV QD-Mini LED 65"" TCL 4K 65C6K","R$ 3.963,60",Via Webcontinental,https://www.buscape.com.br/tv/smart-tv-qd-mini...,Smart TV
4,5,"Smart TV LED 50"" LG 4K UA7500","R$ 2.230,41",Via Magazine Luiza,N/A,Smart TV
